# Multi-Objective BO on Aberrations : c1-a1x-a1y

## Before you start

This notebook runs multi-objective Bayesian optimization of defocus (C1) and twofold astigmatism (A1) on the atomistic gold digital twin. The twin renders each image through the twin corrector's aberrations, so changing C1 and A1 changes the image.

Start the twin stack from the repository root:

    uv run startup_scripts/run_servers.py --yaml configs/polycrystalline_gold.yaml

That starts the Tango database on localhost:9094, Tiled, the gold twin, the twin corrector and the ANALYSIS device.

The optimization cells (2.3 and later) also need `torch` and `botorch`, which the default install does not include.


In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import tango
from tiled.client import from_uri

sys.path.insert(0, "../")
from asyncroscopy.analysis.correction import ceos_to_pytemlib

# The gold twin stack from configs/polycrystalline_gold.yaml.
os.environ["TANGO_HOST"] = "localhost:9094"

microscope = tango.DeviceProxy("asyncroscopy/instrument/default")
scan = tango.DeviceProxy("asyncroscopy/scan/default")
data = tango.DeviceProxy("asyncroscopy/data/default")
corrector = tango.DeviceProxy("asyncroscopy/corrector/default")
analysis = tango.DeviceProxy("asyncroscopy/analysis/default")

for proxy in (microscope, scan, data, corrector, analysis):
    proxy.set_timeout_millis(120_000)
    proxy.ping()
    print(proxy.name(), proxy.state())

client = from_uri(json.loads(data.get_config())["uri"])

In [ ]:
from pyTEMlib import probe_tools as pt

ACCELERATING_VOLTAGE_V = 200e3  # the gold twin's voltage


def read_haadf(key):
    """Read the HAADF image saved under a DATA/Tiled key."""
    node = client[key]
    if "image" in node:
        node = node["image"]
    if "HAADF" in node:
        node = node["HAADF"]
    return np.asarray(node.read(), dtype=np.float64)


def current_aberrations():
    """The twin corrector's aberrations, in CEOS notation (name -> [x, y] in meters)."""
    return json.loads(corrector.get_aberrations_coeff_sim())


def print_aberrations(ceos):
    """Show CEOS aberrations in pyTEMlib's table (pyTEMlib notation, nm)."""
    table = pt.get_target_aberrations("Spectra300", int(ACCELERATING_VOLTAGE_V))
    table.update(ceos_to_pytemlib(ceos))
    table["acceleration_voltage"] = ACCELERATING_VOLTAGE_V
    pt.print_aberrations(table)

### Connections:

In [ ]:
# Check that the corrector and the ANALYSIS device answer.
print(json.loads(corrector.get_info())["result"])
print(json.loads(analysis.get_targets()))

In [ ]:
# Microscope state as the twin reports it.
print(json.loads(microscope.get_parameters()))

In [ ]:
# The corrector's limits, if any are configured.
print(json.loads(corrector.get_correction_limits()))

### Help commands:

In [ ]:
# The commands each device offers.
for proxy in (microscope, corrector, analysis):
    print(proxy.name(), sorted(command.cmd_name for command in proxy.command_list_query()))

In [ ]:
# Details of one command.
print(microscope.command_query("acquire_scanned_image"))

### Setting the aberrations from known values:

In [ ]:
# Start from the twin's aberrations with C1 and A1 set to zero.
aberrations = current_aberrations()
aberrations["C1"] = [0.0]
aberrations["A1"] = [0.0, 0.0]
corrector.set_aberrations_coeff_sim(json.dumps(aberrations))
print_aberrations(aberrations)

In [ ]:
# At any time, view the current aberrations.
print_aberrations(current_aberrations())

### Get an image:

The gold twin renders it through the twin corrector's current aberrations.


In [ ]:
scan.imsize = 512
scan.dwell_time = 10e-6

key = microscope.acquire_scanned_image(["HAADF"])
img = read_haadf(key)

plt.imshow(img, cmap="gray")
plt.title(f"Gold twin HAADF: {key}")
plt.show()

### How it works:

The notebook talks to Tango devices. `microscope.acquire_scanned_image` asks the gold twin for an image. The twin reads the aberrations from the twin corrector, builds the probe with pyTEMlib, and saves the image to Tiled. The ANALYSIS device scores images by their Tiled key.


### try changing an aberration

In [ ]:
# correct_aberration removes the value you send, as on the real CEOS corrector.
# Removing 6 nm from C1 = 0 leaves C1 = -6 nm.
corrector.correct_aberration("C1 6e-9")

In [ ]:
key = microscope.acquire_scanned_image(["HAADF"])
img = read_haadf(key)

plt.imshow(img, cmap="gray")
plt.title(f"Gold twin HAADF after the C1 change: {key}")
plt.show()

## 2. Lets do BO to tune the parameters

### 2.1 set helper functions

In [ ]:
def contrast_rms(im, eps=1e-12):
    m = np.mean(im)
    return np.std(im) / (m + eps)


def set_c1_a1(c10_nm, c12a_nm, c12b_nm):
    """Set C1 and A1 directly, as uploading aberrations did.

    The inputs are pyTEMlib's C10, C12a and C12b in nm. In CEOS notation these
    are C1 and the x and y parts of A1, with the same values, in meters.
    """
    aberrations = current_aberrations()
    aberrations["C1"] = [float(c10_nm) * 1e-9]
    aberrations["A1"] = [float(c12a_nm) * 1e-9, float(c12b_nm) * 1e-9]
    corrector.set_aberrations_coeff_sim(json.dumps(aberrations))


def nyquist_fraction_score(key):
    """Fraction of the Nyquist limit the image resolves (0 to 1), from the ANALYSIS device."""
    result = json.loads(analysis.resolution(key))
    if result["resolution_angstrom"] is None:
        return 0.0, result
    return float(min(1.0, result["nyquist_angstrom"] / result["resolution_angstrom"])), result


def acquire_stem_image_contrast_and_fft(c10, c12a, c12b, plot_diagnostics=False):
    """Set C1 and A1, acquire a HAADF image, and return (contrast, FFT score, image)."""
    set_c1_a1(c10, c12a, c12b)
    key = microscope.acquire_scanned_image(["HAADF"])
    sim_im = read_haadf(key)
    contrast = contrast_rms(sim_im)
    score, result = nyquist_fraction_score(key)

    if plot_diagnostics:
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(sim_im, cmap="gray")
        axes[0].set_title(f"C10={c10:.1f}, C12a={c12a:.1f}, C12b={c12b:.1f} nm")
        centered = sim_im - sim_im.mean()
        axes[1].imshow(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(centered))) ** 2), cmap="hot")
        resolution = result["resolution_angstrom"]
        axes[1].set_title("FFT power (log): " + (f"resolves {resolution:.2f} A" if resolution else "no spots above noise"))
        plt.suptitle(f"contrast={contrast:.3f}, FFT score={score:.3f}")
        plt.tight_layout()
        plt.show()

    return contrast, score, sim_im

### 2.2 Setup seed data for the BO

In [ ]:
# Parameter ranges based on your exploration
param_ranges = {
    'C10': (-8, 8),    # defocus
    'C12a': (-10, 10), # twofold astigmatism (a)
    'C12b': (-10, 10)  # twofold astigmatism (b)
}

In [ ]:
# Create full grid (coarser for computational efficiency)
n_grid = 7  # 7^3 = 343 points
c10_grid = np.linspace(*param_ranges['C10'], n_grid)
c12a_grid = np.linspace(*param_ranges['C12a'], n_grid)
c12b_grid = np.linspace(*param_ranges['C12b'], n_grid)
C10, C12A, C12B = np.meshgrid(c10_grid, c12a_grid, c12b_grid, indexing='ij')
full_grid = np.stack([C10.flatten(), C12A.flatten(), C12B.flatten()], axis=1)

In [ ]:
# Sample seed points
n_seed = 4
rng = np.random.default_rng(42)  # fixed seed: the same seed points every run
seed_indices = rng.choice(len(full_grid), n_seed, replace=False)
seed_points = full_grid[seed_indices]

In [ ]:
# ========== QUERY SEED POINTS ==========
print(f"\nQuerying {n_seed} seed points...")
seed_scores = []
seed_images = []

for i, (c10, c12a, c12b) in enumerate(seed_points):
    contrast, fft_score, sim_im = acquire_stem_image_contrast_and_fft(c10, c12a, c12b, plot_diagnostics=True)
    rewards = np.array((contrast, fft_score))
    seed_scores.append(rewards)
    seed_images.append(sim_im)

seed_scores = np.array(seed_scores)


In [ ]:
plt.scatter(seed_scores[:, 0], seed_scores[:, 1])

### 2.3 Simple MOBO 

In [ ]:
import torch
from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood
import numpy as np
import matplotlib.pyplot as plt


from botorch.acquisition.multi_objective import qExpectedHypervolumeImprovement, qLogExpectedHypervolumeImprovement 
from botorch.utils.multi_objective.box_decompositions import NondominatedPartitioning
from botorch.utils.multi_objective import is_non_dominated
from botorch.utils.transforms import normalize, unnormalize

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# ========== INITIAL SEED POINTS (from previous code) ==========
print(f"Starting with {n_seed} seed points...")
print(f"Best initial contrast: {seed_scores[:, 0].max():.4f}")

# Convert to tensors
train_X = torch.tensor(seed_points, dtype=torch.float64)
train_Y = torch.tensor(seed_scores, dtype=torch.float64)

# Define bounds for optimization
bounds = torch.tensor([
    [param_ranges['C10'][0], param_ranges['C12a'][0], param_ranges['C12b'][0]],  # lower bounds
    [param_ranges['C10'][1], param_ranges['C12a'][1], param_ranges['C12b'][1]]   # upper bounds
], dtype=torch.float64)



In [ ]:
import torch

# ---- device & dtype ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64

torch.set_default_dtype(dtype)

# move inputs/bounds to device+dtype
train_X = train_X.to(device=device, dtype=dtype)
train_Y = train_Y.to(device=device, dtype=dtype)
bounds  = bounds.to(device=device, dtype=dtype)

n_bo_steps = 50
all_X = train_X.clone()
all_Y = train_Y.clone()
all_images = seed_images.copy()

print("\n" + "="*60)
print("Starting Multi-Objective Bayesian Optimization with EHVI")
print("="*60)

ref_point = train_Y.min(dim=0).values - 0.1 * train_Y.std(dim=0)
print(f"Reference point: {ref_point.detach().cpu().numpy()}")

for step in range(n_bo_steps):
    print(f"\n--- BO Step {step + 1}/{n_bo_steps} ---")
    
    # Train GP (model follows tensor's device/dtype)
    print("Training Multi-Output GP...")
    gp_model = SingleTaskGP(
        all_X, all_Y,
        input_transform=Normalize(d=all_X.shape[-1]).to(device=device, dtype=dtype),
        outcome_transform=Standardize(m=all_Y.shape[-1]).to(device=device, dtype=dtype),
    ).to(device=device, dtype=dtype)
    
    gp_model.likelihood.noise_covar.initialize(noise=0.01)
    mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model).to(device=device, dtype=dtype)
    fit_gpytorch_mll(mll)
    
    # EHVI acquisition
    print("Computing Pareto frontier...")
    pareto_mask = is_non_dominated(all_Y)
    pareto_Y = all_Y[pareto_mask]
    print(f"Pareto frontier size: {pareto_Y.shape[0]}")
    
    partitioning = NondominatedPartitioning(ref_point=ref_point, Y=pareto_Y)
    EHVI = qLogExpectedHypervolumeImprovement(
        model=gp_model,
        ref_point=ref_point.tolist(),
        partitioning=partitioning,
    )
    
    # Optimize
    print("Optimizing acquisition function...")
    candidate, acq_value = optimize_acqf(
        acq_function=EHVI,
        bounds=bounds,
        q=1,
        num_restarts=10,
        raw_samples=100,
    )
    
    next_X = candidate.detach()
    next_params = next_X.squeeze().detach().cpu().numpy()
    print(f"EHVI value: {acq_value:.6f}")
    
    # Query simulator (returns CPU values)
    print("Querying STEM simulator...")
    objective1, objective2, next_image = acquire_stem_image_contrast_and_fft(
        next_params[0], next_params[1], next_params[2], plot_diagnostics=True
    )
    next_Y = torch.tensor([[objective1, objective2]], dtype=dtype, device=device)
    
    print(f"Observed objectives: [{objective1:.4f}, {objective2:.4f}]")
    
    # Update tensors on-device
    all_X = torch.cat([all_X, next_X], dim=0)
    all_Y = torch.cat([all_Y, next_Y], dim=0)
    all_images.append(next_image)
    
    new_pareto_mask = is_non_dominated(all_Y)
    if new_pareto_mask[-1]:
        print("✓ NEW PARETO POINT!")


### 2.4 Lets Visualize the Pareto-Frontier and see the MO-BO suggested solutions on it

In [ ]:
# ========== FIND EXTREME AND MID PARETO POINTS ==========
print("\n" + "="*60)
print("Pareto Frontier Analysis")
print("="*60)

final_pareto_mask = is_non_dominated(all_Y)
final_pareto_X = all_X[final_pareto_mask]
final_pareto_Y = all_Y[final_pareto_mask]
pareto_indices = torch.where(final_pareto_mask)[0].cpu().numpy()

print(f"Number of Pareto optimal points: {final_pareto_Y.shape[0]}")

# Find extreme points
extreme_indices = []

# Extreme for Objective 1
max_obj1_idx = torch.argmax(final_pareto_Y[:, 0]).item()
min_obj1_idx = torch.argmin(final_pareto_Y[:, 0]).item()

# Extreme for Objective 2
max_obj2_idx = torch.argmax(final_pareto_Y[:, 1]).item()
min_obj2_idx = torch.argmin(final_pareto_Y[:, 1]).item()

extreme_indices.extend([max_obj1_idx, min_obj1_idx, max_obj2_idx, min_obj2_idx])
extreme_indices = list(set(extreme_indices))  # Remove duplicates

# Find middle point (balanced trade-off)
# Normalize objectives to [0,1] then find point closest to (0.5, 0.5)
normalized_pareto_Y = (final_pareto_Y - final_pareto_Y.min(dim=0).values) / (final_pareto_Y.max(dim=0).values - final_pareto_Y.min(dim=0).values + 1e-8)
distances_to_center = torch.norm(normalized_pareto_Y - 0.5, dim=1)
mid_idx = torch.argmin(distances_to_center).item()

# Combine: extremes + mid
selected_indices = sorted(list(set(extreme_indices + [mid_idx])))

print(f"\nSelected Pareto points for visualization: {len(selected_indices)}")
for idx in selected_indices:
    pareto_idx = pareto_indices[idx]
    params = all_X[pareto_idx].cpu().numpy()
    obj1, obj2 = all_Y[pareto_idx, 0].item(), all_Y[pareto_idx, 1].item()
    
    label = ""
    if idx == max_obj1_idx:
        label += "[MAX Obj1] "
    if idx == min_obj1_idx:
        label += "[MIN Obj1] "
    if idx == max_obj2_idx:
        label += "[MAX Obj2] "
    if idx == min_obj2_idx:
        label += "[MIN Obj2] "
    if idx == mid_idx:
        label += "[MID/Balanced] "
    
    print(f"  {label}")
    print(f"    Obj1={obj1:.4f}, Obj2={obj2:.4f}")
    print(f"    C10={params[0]:.2f}, C12a={params[1]:.2f}, C12b={params[2]:.2f}")

# ========== VISUALIZE ONLY EXTREME + MID PARETO IMAGES ==========
n_selected = len(selected_indices)
n_cols = min(3, n_selected)
n_rows = int(np.ceil(n_selected / n_cols))

fig = plt.figure(figsize=(7*n_cols, 7*n_rows))

for plot_idx, pareto_idx_in_frontier in enumerate(selected_indices):
    ax = fig.add_subplot(n_rows, n_cols, plot_idx + 1)
    
    pareto_idx = pareto_indices[pareto_idx_in_frontier]
    img = all_images[pareto_idx]
    params = all_X[pareto_idx].cpu().numpy()
    obj1, obj2 = all_Y[pareto_idx, 0].item(), all_Y[pareto_idx, 1].item()
    
    # Determine label
    label = ""
    if pareto_idx_in_frontier == max_obj1_idx:
        label = "MAX Obj1"
        color = 'red'
    elif pareto_idx_in_frontier == min_obj1_idx:
        label = "MIN Obj1"
        color = 'blue'
    elif pareto_idx_in_frontier == max_obj2_idx:
        label = "MAX Obj2"
        color = 'green'
    elif pareto_idx_in_frontier == min_obj2_idx:
        label = "MIN Obj2"
        color = 'orange'
    elif pareto_idx_in_frontier == mid_idx:
        label = "BALANCED (Mid)"
        color = 'purple'
    else:
        label = "Extreme"
        color = 'black'
    

    ax.imshow(np.array(img), cmap='gray')
    ax.set_title(
        f'{label}\n'
        f'Obj1={obj1:.4f}, Obj2={obj2:.4f}\n'
        f'C10={params[0]:.1f}, C12a={params[1]:.1f}\n'
        f'C12b={params[2]:.1f}\n',
        fontsize=12,
        fontweight='bold',
        color=color
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('pareto_extreme_mid_images.png', dpi=150, bbox_inches='tight')
plt.show()

# ========== COMBINED RESULTS PLOT ==========
fig = plt.figure(figsize=(18, 6))

# 1. Objective space
ax1 = plt.subplot(1, 3, 1)
ax1.scatter(all_Y[:, 0].cpu().numpy(), all_Y[:, 1].cpu().numpy(), 
           c='lightblue', s=150, alpha=0.6, edgecolors='gray',
           label='All evaluations')
ax1.scatter(final_pareto_Y[:, 0].cpu().numpy(), final_pareto_Y[:, 1].cpu().numpy(), 
           c='lightcoral', s=200, alpha=0.5, edgecolors='black', 
           linewidths=1, label='Pareto frontier')

# Highlight extreme and mid points
colors = []
labels_legend = []
for idx in selected_indices:
    if idx == max_obj1_idx:
        colors.append('red')
        if 'MAX Obj1' not in labels_legend:
            labels_legend.append('MAX Obj1')
    elif idx == max_obj2_idx:
        colors.append('green')
        if 'MAX Obj2' not in labels_legend:
            labels_legend.append('MAX Obj2')
    elif idx == mid_idx:
        colors.append('purple')
        if 'Balanced' not in labels_legend:
            labels_legend.append('Balanced')
    else:
        colors.append('orange')

for idx, color in zip(selected_indices, colors):
    ax1.scatter(final_pareto_Y[idx, 0].cpu().numpy(), final_pareto_Y[idx, 1].cpu().numpy(),
               c=color, s=400, marker='*', edgecolors='black', linewidths=2, zorder=10)

ax1.set_xlabel('Objective 1 (Contrast)', fontsize=12)
ax1.set_ylabel('Objective 2: FFT score (fraction of Nyquist)', fontsize=12)
ax1.set_title('Pareto Frontier (★ = Extreme/Mid points)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# 2. BO progress
ax2 = plt.subplot(1, 3, 2)
iterations = range(len(all_Y))
ax2.plot(iterations, all_Y[:, 0].cpu().numpy(), 'o-', label='Objective 1', 
        alpha=0.7, linewidth=2, markersize=8)
ax2.plot(iterations, all_Y[:, 1].cpu().numpy(), 's-', label='Objective 2', 
        alpha=0.7, linewidth=2, markersize=8)
ax2.axvline(len(train_Y)-1, color='red', linestyle='--', 
          label='BO start', linewidth=2)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Objective Value', fontsize=12)
ax2.set_title('BO Progress', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# 3. Hypervolume
from botorch.utils.multi_objective.hypervolume import Hypervolume

hv_computer = Hypervolume(ref_point=ref_point)
hypervolumes = []
for i in range(len(train_Y), len(all_Y) + 1):
    current_Y = all_Y[:i]
    pareto_mask_i = is_non_dominated(current_Y)
    pareto_Y_i = current_Y[pareto_mask_i]
    hv = hv_computer.compute(pareto_Y_i)
    hypervolumes.append(hv)

ax3 = plt.subplot(1, 3, 3)
ax3.plot(range(len(train_Y), len(all_Y) + 1), hypervolumes, 'o-', 
        linewidth=2, markersize=8, color='purple')
ax3.set_xlabel('Iteration', fontsize=12)
ax3.set_ylabel('Hypervolume', fontsize=12)
ax3.set_title('Hypervolume Improvement', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('multi_objective_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== SUMMARY ===")
print(f"Total evaluations: {len(all_Y)}")
print(f"Pareto frontier size: {len(pareto_indices)}")
print(f"Extreme + Mid points shown: {len(selected_indices)}")
print(f"Final hypervolume: {hypervolumes[-1]:.4f}")